In [2]:
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import statsmodels.formula.api as smf
from pathlib import Path

import sys; sys.path.insert(0, '..')
from src.palette import register, content_colors, colorway, CONTENT_ORDER

CONTENT_COLORS = register('light')

In [3]:
conn = sqlite3.connect('../data/lafc_content.db')

## Date Filter

Filtered to 2024 onward. LAFC's format strategy changed in 2024Q1 — Shorts went from 3.9% of uploads to 26.4% and stayed there, and quarterly upload volume roughly tripled. Shorts barely existed prior to 2024, and this shows a large strategy shift, so I filter to it. See strategy_exploration.ipynb for the quarterly breakdown.

In [4]:
with open('../sql/videos_vs_lafc_match_context.sql') as f:
    query = f.read()

df = pd.read_sql(query, conn)
df = df[df['published_at'] >= '2024-01-01T00:00:00Z'].copy()
df['content_type'] = df['content_type'].fillna('no_playlist')
df['playlist'] = df['playlist'].fillna('(no playlist)')

## Distributions

Histograms of views, and engagement rate.

In [5]:
fig = px.histogram(
    df, x='view_count',
    title="Videos per view - linear scale"
    )

fig.update_xaxes(title='Views')

fig.update_yaxes(title='Number of videos')

fig.show()

In [6]:
print('View Count Mean:', df['view_count'].mean())
print('View Count Median:', df['view_count'].median())


View Count Mean: 18477.02817622951
View Count Median: 1802.0


In [7]:
df['log10_views'] = np.log10(df['view_count'])

fig = px.histogram(
    df, x='log10_views',
    nbins=80,
    title='Videos per View - Log Scale', 
    )
fig.update_xaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'],
    title='Views')

fig.update_yaxes(
    title='Number of videos'
)

fig.show()

In [8]:
fig = px.histogram(
    df, x='engagement_rate',
    nbins=80,
    title= 'Engagement rate per video'
    )

fig.update_yaxes()

fig.update_xaxes(tickformat='.1%')

fig.show()

In [9]:
table = df[['title', 'format', 'content_type', 'engagement_rate']].sort_values('engagement_rate', ascending=False)
display(table.head())

,title,format,content_type,engagement_rate
614,Acción LAFC Con Armando Aguayo | Ep. 72,horizontal,podcast,0.18006
1,LAFC vs QRO | Postmatch Media,horizontal,press_interview,0.15686
1727,2024 Pride Recap,horizontal,community,0.14839
628,Acción LAFC Con Armando Aguayo | Ep. 71,horizontal,podcast,0.14364
664,LAFC+ | Ep. 68,horizontal,podcast,0.14206


View counts had a huge span. Some videos had less than 100 views, while a handful had more than 1 million. The median was 2,843, but a few high-view videos pushed the mean up to 28,065. After the first histogram, I switched to a charting the view count on a log 10 scale, to better visualize the data. In later charts I used view count median rather than mean.

Engagement rate had a much less dramatic span, so I was able to chart without using a log scale, choosing to display percentage to one decimal point.

But I realized that the longer a video is up on Youtube, the longer it has to accumulate views. So I decided to take a look at that effect.

In [10]:
# Add time since published to df

snapshot_row = pd.read_sql('SELECT MAX(fetched_at) AS snapshot_at FROM videos', conn)
SNAPSHOT_AT = pd.Timestamp(snapshot_row['snapshot_at'][0])

published = pd.to_datetime(df['published_at'], format='ISO8601', utc=True)

# How long this video has had to collect views.
df['age_days'] = (SNAPSHOT_AT - published).dt.total_seconds() / 86_400

# Which quarter it went out in. Stands in for how big the channel was then.
df['published_quarter'] = published.dt.to_period('Q').astype(str)

print(f'Snapshot taken {SNAPSHOT_AT:%Y-%m-%d}')
print(f'Video ages run {df["age_days"].min():.0f} to {df["age_days"].max():.0f} days')

df

Snapshot taken 2026-08-13
Video ages run 0 to 954 days


/var/folders/g4/h7hn6rpd2hv1lz_lqhhb8qzw0000gn/T/ipykernel_43031/578316147.py:12: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['published_quarter'] = published.dt.to_period('Q').astype(str)


,video_id,title,description,published_at,duration,view_count,like_count,comment_count,engagement_rate,format,...,lafc_played,lafc_wins,opp_points,opp_played,opp_wins,days_since_match,days_until_match,log10_views,age_days,published_quarter
0,IxrFLgowFd4,Armindo Sieb is Black & Gold.,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T16:40:02Z,PT52S,275,23,7,0.10909,horizontal,...,18.0,10.0,33.0,16.0,10.0,11.72,NaN,2.439333,0.086067,2026Q3
1,HugEGKBw0kk,LAFC vs QRO | Postmatch Media,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T09:02:51Z,PT14M19S,306,22,26,0.15686,horizontal,...,18.0,10.0,33.0,16.0,10.0,11.40,NaN,2.485721,0.403555,2026Q3
2,pLVoxNTyGLI,The top scorer in Leagues Cup history 📈,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:30:24Z,PT15S,6287,169,12,0.02879,short,...,18.0,10.0,33.0,16.0,10.0,11.33,NaN,3.798443,0.467756,2026Q3
3,wz6UrdGQWjY,BOUANGA EQUALIZER 💥,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:15:01Z,PT13S,3940,91,4,0.02411,short,...,18.0,10.0,33.0,16.0,10.0,11.32,NaN,3.595496,0.478439,2026Q3
4,SqgJkPzCN6Y,Denis Bouanga equalizes against Querétaro,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:02:06Z,PT13S,1712,55,6,0.03563,horizontal,...,18.0,10.0,33.0,16.0,10.0,11.31,NaN,3.233504,0.487409,2026Q3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1947,nl2vg8fuuNs,John Thorrington | 2024 Preseason Press Confer...,LAFC Co-President & General Manager John Thorr...,2024-01-26T01:33:57Z,PT30M32S,2921,112,6,0.04040,horizontal,...,33.0,14.0,47.0,33.0,12.0,96.02,29.83,3.465532,930.715291,2024Q1
1948,Xid_keBHB8w,Season 7 ⏳,Subscribe to the LAFC YouTube Channel: https:/...,2024-01-21T02:25:24Z,PT29S,2492,137,5,0.05698,short,...,33.0,14.0,47.0,33.0,12.0,91.06,34.79,3.396548,935.679562,2024Q1
1949,oiLKSEbbSK0,Part of our History | Maxime Crépeau,Subscribe to the LAFC YouTube Channel: https:/...,2024-01-19T21:26:01Z,PT3M46S,1218,59,9,0.05583,horizontal,...,33.0,14.0,47.0,33.0,12.0,89.85,36.00,3.085647,936.887467,2024Q1
1950,eoVVpIqWaW0,Dénis Bouanga | All 2023 Goals,Every single goal from Dénis Bouanga's Golden ...,2024-01-03T01:30:24Z,PT8M45S,32847,474,44,0.01577,horizontal,...,33.0,14.0,47.0,33.0,12.0,73.02,52.83,4.516496,953.717756,2024Q1


In [11]:
# Views accumulate, so older videos have had longer to collect them.
# Check whether that shows up in the data.
print('Pearson,  log10 views vs age :', round(df['log10_views'].corr(df['age_days']), 3))
print('Spearman, rank        vs age :',
      round(df['view_count'].corr(df['age_days'], method='spearman'), 3))

Pearson,  log10 views vs age : -0.411
Spearman, rank        vs age : -0.448


But the correlation coefficient of log10 views vs. age in age in days (since upload) was negative. The newer videos were seemingly getting more views. I decided to look by quarter.

In [12]:
df.groupby('published_quarter')['view_count'].agg(n='size', median_views='median')

,n,median_views
published_quarter,,
2024Q1,53,2323.0
2024Q2,169,501.0
2024Q3,333,622.0
2024Q4,182,641.5
2025Q1,150,1348.5
2025Q2,244,1418.5
2025Q3,309,8103.0
2025Q4,124,4152.0
2026Q1,114,3817.0


Median views were increasing over time, and there was a noticeable bump in Q3 of 2025. A strange finding I decided to come back to later.

## Format and content type

I used the tab separation to separate the videos into shorts, horizontal, and live videos. It's all YouTube, but I thought shorts vs horizontal / live categories was a medium differentiator. The shorts are all verticals, much shorter duration, and consumed in a scroll, whereas the horizontal are the older YouTube format.

Through various explorations of the data, I arrived at using playlist categorization to define content type -- a category I placed within the larger format category.

In [13]:
df['format'].value_counts()

format
horizontal    1342
short          528
live            82
Name: count, dtype: int64

In [14]:
df.groupby('format')['view_count'].median().sort_values(ascending=False)

format
short         6291.0
live          2023.0
horizontal     914.0
Name: view_count, dtype: float64

In [15]:
df['playback_type'] = np.where(df['format'] == 'short', 'short', 'horizontal+live')
df['is_short'] = df['format'] == 'short'

order = ['short', 'horizontal+live']
n = df['playback_type'].value_counts()

fig = px.box(
    df, x='playback_type', y='view_count',
    log_y=True,
    color='playback_type',
    category_orders={'playback_type': order},
    title='View Count by Playback Type - Short vs. Horizontal+Live',
    labels={'playback_type': '', 'view_count': 'View Count'},
)

fig.update_traces(marker=dict(opacity=0.3, size=4), jitter=0.4)

fig.update_layout(showlegend=False)

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[100, 1_000, 10_000, 100_000, 1_000_000],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [16]:
order = ['short', 'horizontal+live']
n = df['playback_type'].value_counts()

fig = px.box(
    df, x='playback_type', y='engagement_rate',
    color='playback_type',
    category_orders={'playback_type': order},
    title='Engagement Rate by Playback Type - Short vs. Horizontal+Live',
    labels={'playback_type': '', 'engagement_rate': 'Engagement Rate'},
)

fig.update_traces(marker=dict(opacity=0.3, size=4), jitter=0.4)

fig.update_layout(showlegend=False)

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

**Shorts get views. Horizontal / Live Gets Engagement**

Shorts get ×4.7 the median views of horizontal+live (8,331 vs 1,757) and 1.2 points *less* engagement (4.30% vs 5.52%).

In [17]:
table = pd.crosstab(df['format'], df['content_type'])
table

content_type,community,feature,full_match,highlights,match_preview,no_playlist,podcast,press_interview,show,unclassified
format,,,,,,,,,,
horizontal,9,20,7,264,71,90,303,337,148,93
live,0,0,2,0,1,6,69,1,0,3
short,0,0,0,46,1,452,0,0,0,29


In [18]:
shorts_df = df[df['format'] == 'short'].copy()
playlist_table = shorts_df['playlist'].value_counts(dropna=False)
content_type_table = shorts_df['content_type'].value_counts(dropna=False)

display(content_type_table)
display(playlist_table)

content_type
no_playlist      452
highlights        46
unclassified      29
match_preview      1
Name: count, dtype: int64

playlist
(no playlist)        452
Highlights            46
The Son Spotlight     29
Match Previews         1
Name: count, dtype: int64

In [19]:
order = shorts_df.groupby('content_type')['view_count'].median().sort_values().index.tolist()
n = shorts_df['content_type'].value_counts()

fig = px.box(
    shorts_df, x='content_type', y='log10_views',
    color='content_type',
    category_orders={'content_type': order},
    title='Views (log 10) by Content Type - Short Format',
    labels={'content_type': 'Content Type', 'log10_views': "Views"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [20]:
order = shorts_df.groupby('content_type')['engagement_rate'].median().sort_values().index.tolist()
n = shorts_df['content_type'].value_counts()

fig = px.box(
    shorts_df, x='content_type', y='engagement_rate',
    color='content_type',
    category_orders={'content_type': order},
    title='Engagement Rate (log 10) by Content Type - Short Format',
    labels={'content_type': 'Content Type', 'engagment_rate': "Engagement Rate"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

In [21]:
unclassified_shorts_df = shorts_df[shorts_df['content_type'] == 'unclassified'].copy()
unclassified_shorts_df['playlist'].value_counts(dropna=False)

playlist
The Son Spotlight    29
Name: count, dtype: int64

The content type label turned out to not be imporant in shorts, as there were only four content types in shorts: highlights, no playlist, unclassified, and match preview (n=1.) However, within the shorts format -- the unclassified content type was outperforming the rest. Turns out those were all from the Son Spotlight playlist.

In [22]:
horizontal_df = df[df['format'] != 'short'].copy()

order = horizontal_df.groupby('content_type')['view_count'].median().sort_values().index.tolist()
n = horizontal_df['content_type'].value_counts()

fig = px.box(
    horizontal_df, x='content_type', y='log10_views',
    color='content_type',
    color_discrete_map=CONTENT_COLORS,
    category_orders={'content_type': order},
    title='Views (log 10) by Content Type - Horizontal / Live ',
    labels={'content_type': 'Content Type', 'log10_views': "Views"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [23]:
order = horizontal_df.groupby('content_type')['engagement_rate'].median().sort_values().index.tolist()
n = horizontal_df['content_type'].value_counts()

fig = px.box(
    horizontal_df, x='content_type', y='engagement_rate',
    color='content_type',
    color_discrete_map=CONTENT_COLORS,
    category_orders={'content_type': order},
    title='Engagement Rate by Content Type - Horizontal / Live',
    labels={'content_type': 'Content Type', 'engagement_rate': "Engagement Rate"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

In [24]:
unclassified_horizontal_df = horizontal_df[horizontal_df['content_type'] == 'unclassified'].copy()
unclassified_horizontal_df['playlist'].value_counts(dropna=False)

playlist
The Son Spotlight                  40
Adventures in the U.S. Open Cup    28
Major News                         11
The Vela Vault                      9
The LAFC Academy                    4
Best of LAFC                        2
LAFC's MLS All-Stars                1
LAFC Essentials                     1
Name: count, dtype: int64

In [25]:
son_spotlight_df = unclassified_horizontal_df[unclassified_horizontal_df['playlist'] == 'The Son Spotlight'].copy()
son_spotlight_df[['title', 'playlist']]

,title,playlist
20,One Year of Sonny Goals,The Son Spotlight
36,Sonny vs. SKC | EVERY ANGLE,The Son Spotlight
52,SONNY SCORES HIS 3RD GOAL IN 3 MATCHES | LAFC ...,The Son Spotlight
54,Sonny vs RSL | EVERY ANGLE,The Son Spotlight
67,SONNY SCORES FROM THE TOP OF THE BOX | LAFC vs...,The Son Spotlight
79,Son Heung-Min | EVERY ANGLE of his derby goal ...,The Son Spotlight
92,SONNY’S FIRST GOAL OF THE SEASON | LAG vs LAFC,The Son Spotlight
112,Who has the most aura? w/ Sonny | Ryan & Aaron...,The Son Spotlight
120,Sonny bobblehead arrives in Los Angeles | LAFC...,The Son Spotlight
236,Sonny's goal from the stands | LAFC vs Cruz Azul,The Son Spotlight


The content type label within the horizonal / live format revealed that like shorts vs. horizontal, highlights and match-play focused content types had the hightest median views, but the pattern flipped for engagement rate, where podcast performed the best.

Similarly to the shorts format, one content type, unclassified, peformed well in both metrics, and once again it was the Son Spotlight.

## The Son Spotlight

I took a look at the Son Spotlight, after noticing the high performance of the playlist in both formats.

In [26]:
df['is_son'] = df['playlist'] == 'The Son Spotlight'

df.groupby('is_son').agg(
    n=('view_count', 'size'),
    median_views=('view_count', 'median'),
    median_engagement=('engagement_rate', 'median'))

,n,median_views,median_engagement
is_son,,,
False,1883,1657.0,0.04918
True,69,84187.0,0.05727


In [27]:
df.groupby(['playback_type', 'is_son']).agg(
    n=('view_count', 'size'),
    median_views=('view_count', 'median'), 
    median_engagement=('engagement_rate', 'median'))

n  median_views  median_engagement
playback_type   is_son                                       
horizontal+live False   1384         910.0           0.050285
                True      40       60030.0           0.077960
short           False    499        5810.0           0.045750
                True      29      164231.0           0.053720

In [28]:
df[df['playlist'].isin(['The Son Spotlight', 'The Vela Vault'])].groupby('playlist').agg(
    n=('view_count', 'size'),
    median_views=('view_count', 'median'))

,n,median_views
playlist,,
The Son Spotlight,69,84187.0
The Vela Vault,9,3877.0


**Son is beast.** Son Spotlight videos get x66 horizonal/live median views, and x28 Shorts median views. I take a look at this later with regression.

**That being said...** Is it Son himself or just the format of a player focused playlist? The Vela Vault is the only control and only has 9 videos in it. My gut says it's Son, but I don't have an experiment to back that up.

In [29]:
order = horizontal_df.groupby('playlist')['view_count'].median().sort_values().index.tolist()
n = horizontal_df['playlist'].value_counts()

fig = px.box(
    horizontal_df, y='playlist', x='view_count',
    log_x=True,
    color='playlist',
    category_orders={'playlist': order},
    title='View Count (Log Scale) by Playlist - Horizontal / Live',
    labels={'playlist': 'Playlist', 'view_count': 'View Count'},
    height=600,
)

fig.update_layout(showlegend=False)

fig.update_yaxes(
    tickvals=order,
    ticktext=[f'{p}  (n={n[p]})' for p in order])

fig.update_xaxes(
    tickvals=[100, 1_000, 10_000, 100_000, 1_000_000],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()


In [30]:
order = horizontal_df.groupby('playlist')['engagement_rate'].median().sort_values().index.tolist()
n = horizontal_df['playlist'].value_counts()

fig = px.box(
    horizontal_df, y='playlist', x='engagement_rate',
    color='playlist',
    category_orders={'playlist': order},
    title='Engagement Rate by Playlist - Horizontal / Live',
    labels={'playlist': 'Playlist', 'engagement_rate': 'Engagement Rate'},
    height=600,
)

fig.update_layout(showlegend=False)

fig.update_yaxes(
    tickvals=order,
    ticktext=[f'{p}  (n={n[p]})' for p in order])

fig.update_xaxes(tickformat='.1%')

fig.show()


Looking at median views and engagement rate within the horizontal / live format by playlist type, we can see the Son effect even more clearly. 

We also see the engagement rate rankings of podcasts -- and the top podcast in terms of engagement rate is **Korean** language.

## Timing

Splitting up the videos based on their upload time relative to match kickoff time. Then binning the videos based on days before or after kickoff, and excluding the off-season videos.

In [31]:
# Assigns the two columns into two series.
after  = df['days_since_match']    # days SINCE the previous match (always ≥ 0)
before = df['days_until_match']    # days UNTIL the next match     (always ≥ 0)


# Fills in na with infinity, then compares the two series, row by row, and returns a boolean. If before is smaller - then it returns true, meaning this row is closer to the NEXT match.
closer_to_next = before.fillna(np.inf) < after.fillna(np.inf)

# Assigns either a -before or after based on the boolean in closer_to_next
df['days_from_match'] = np.where(closer_to_next, -before, after)

#Overwrite nan if its been 21 days since (and until) the closest matches

OFFSEASON_DAYS = 21
not_in_cycle = ((after.fillna(np.inf)  > OFFSEASON_DAYS) &
                (before.fillna(np.inf) > OFFSEASON_DAYS))

df.loc[not_in_cycle, 'days_from_match'] = np.nan

print('not in a cycle :', not_in_cycle.sum())
print('before a match :', (df['days_from_match'] < 0).sum())
print('after a match  :', (df['days_from_match'] > 0).sum())

not in a cycle : 125
before a match : 793
after a match  : 1034


In [32]:
#Binning and adding the bin info back to the df.

CYCLE_EDGES  = [-np.inf, -4, -3, -2, -1, 0, 1, 2, 3, 4, np.inf]
CYCLE_LABELS = ['4+ before', '3-4 before', '2-3 before', '1-2 before', '0-1 before',
                '0-1 after', '1-2 after', '2-3 after', '3-4 after', '4+ after']

df['cycle_bin'] = pd.cut(
    df['days_from_match'],
    bins=CYCLE_EDGES,
    labels=CYCLE_LABELS, right=False
    )

display(df[['title', 'days_since_match', 'days_until_match', 'cycle_bin']].sort_values('days_since_match'))

,title,days_since_match,days_until_match,cycle_bin
52,SONNY SCORES HIS 3RD GOAL IN 3 MATCHES | LAFC ...,0.02,6.85,0-1 after
67,SONNY SCORES FROM THE TOP OF THE BOX | LAFC vs...,0.02,2.98,0-1 after
811,Black & Gold Insider Ep. 21 | Christian Beceri...,0.02,2.98,0-1 after
524,Bouanga is on FIRE 🔥,0.03,7.99,0-1 after
523,Denis Bouanga buries it from outside the box v...,0.03,7.99,0-1 after
...,...,...,...,...
1159,The guys reacting to Denis Bouanga's bobblehead,125.10,0.76,0-1 before
1158,Starboy Season Loading ⏳,125.16,0.70,0-1 before
1931,Together We Win In The Stands.,125.69,0.16,0-1 before
347,The season starts today 🏟️,125.73,0.33,0-1 before


In [33]:
print(df['cycle_bin'].value_counts(dropna=False).sort_index())

cycle_bin
4+ before     178
3-4 before     85
2-3 before    168
1-2 before    217
0-1 before    145
0-1 after     480
1-2 after     137
2-3 after      98
3-4 after      72
4+ after      247
NaN           125
Name: count, dtype: int64


In [34]:
PLOT_BINS = CYCLE_LABELS[1:-1]      #Drops first and last bins, since they are farthest away from kickoffs

KICKOFF_X = PLOT_BINS.index('0-1 after') - 0.5   #Sets kickoff line on charts below

# Drops off-season videos and videos with no engagment rate (although none exist, but just in case)

plot_df = df[df['cycle_bin'].isin(PLOT_BINS)].dropna(subset=['engagement_rate']).copy()
plot_df['cycle_bin'] = plot_df['cycle_bin'].cat.remove_unused_categories()
n = plot_df['cycle_bin'].value_counts()

In [35]:
fig = px.box(
    plot_df, x='cycle_bin', y='view_count',
    log_y=True,
    category_orders={'cycle_bin': PLOT_BINS},
    color='cycle_bin',
    points=False,
    title='View Count Across the Match Cycle',
    labels={'cycle_bin': 'Position in match cycle',
            'view_count': 'View Count (log)'},
)

fig.update_xaxes(
    tickvals=PLOT_BINS,
    ticktext=[f'{b}<br>n={n[b]}' for b in PLOT_BINS])

fig.add_vline(x=KICKOFF_X, line_dash='dot', line_color='#888888',
              annotation_text='kickoff', annotation_position='top')

fig.show()

In [36]:
fig = px.box(
    plot_df, x='cycle_bin', y='engagement_rate',
    color='cycle_bin',
    category_orders={'cycle_bin': PLOT_BINS},
    points=False,
    title='Engagement Rate Across the Match Cycle',
    labels={'cycle_bin': 'Position in match cycle',
            'engagement_rate': 'Engagement Rate'},
)

fig.update_xaxes(
    tickvals=PLOT_BINS,
    ticktext=[f'{b}<br>n={n[b]}' for b in PLOT_BINS])

fig.update_yaxes(tickformat='.1%')

fig.add_vline(x=KICKOFF_X, line_dash='dot', line_color='#888888',
              annotation_text='kickoff', annotation_position='top')

fig.show()

In [37]:
table = plot_df.groupby('cycle_bin', observed=True).agg(
        n=('view_count', 'size'),
        median_views=('view_count', 'median'),
        median_engagement_rate=('engagement_rate', 'median'))

table['median_engagement_rate'] = (table['median_engagement_rate'] * 100).round(2).astype(str) + '%'

table

,n,median_views,median_engagement_rate
cycle_bin,,,
3-4 before,85,1801.0,4.79%
2-3 before,168,915.0,5.45%
1-2 before,217,1212.0,5.17%
0-1 before,145,1593.0,5.64%
0-1 after,480,2975.5,3.99%
1-2 after,137,1912.0,5.32%
2-3 after,98,1357.0,5.17%
3-4 after,72,1611.0,4.83%


**Views spike immediately after kickoff** The 0-1 day after category had the highest number of views after flat median views.

**Engagement peaks the day leading up to kickoff.** 

## Controlling for how long a video has been up

Back to the negative correlation from the Distributions section. Before the regression, I wanted to work out what was happening to views related to upload date. I had found a negative correlation between days since upload and view count, and noticed a jump in 2025-Q3 in particular.

I broke out a few content types to see if this pattern was true across different content types.

In [46]:
# Looking at median views and engagement within content type
for content_type in ['press_interview', 'podcast', 'highlights']:
    one_type = df[df['content_type'] == content_type]
    print(f'\n{content_type}')
    print(one_type.groupby('published_quarter').agg(
        n=('view_count', 'size'),
        median_views=('view_count', 'median'),
        median_engagement=('engagement_rate', 'median')))


press_interview
                     n  median_views  median_engagement
published_quarter                                      
2024Q2              31         278.0           0.041540
2024Q3             103         402.0           0.045450
2024Q4              49         340.0           0.042860
2025Q1              26         664.0           0.047100
2025Q2              34         718.0           0.045790
2025Q3              32        1716.5           0.057485
2025Q4               9        3166.0           0.060240
2026Q1              16        2786.5           0.072600
2026Q2              26        1871.0           0.051955
2026Q3              12        1655.5           0.053260

podcast
                    n  median_views  median_engagement
published_quarter                                     
2024Q1             17         535.0           0.058820
2024Q2             39         333.0           0.052080
2024Q3             34         355.5           0.052670
2024Q4             29      

All three step up between 2025Q2 and 2025Q3 for median view count. Highlights did not step up for engagement rate, but that's no unusual for this content type. I realized this must be related to the **Son Effect** because he was signed August 6, 2025. The other option was that the format strategy shift had changed median view count over time, but I ruled that out as the strategy shift happened at the beginning of 2024.

It looks like the signing of Son lifted the audience across content types.

## Regression

My analysis was showing that short vs. horizonal / live, the Son effect, and published timing were having an effect on median view count, as well as engagement rate, so I decided to run some regression models to determine how much. 

I also needed to incorporate a way to control for the increased audience that seemed to accompany Son's signing in 2025-Q3. Because videos that weren't from the Son Spotlight also saw increased metrics from that point forward. Otherwise, the regression models wouldn't recognize that something shifted at that time.

### OLS: log₁₀ views on format, content type, and match-cycle position
Baseline: a long-form, highlights video published 0–1 days before a match

In [54]:
# Dropped on sample size (community n=9, full_match n=9).
TOO_SMALL = ['community', 'full_match']

# Also drop videos under 30 days old - they haven't finished accumulating views.
model_df = df[(df['age_days'] >= 30) & ~df['content_type'].isin(TOO_SMALL)].copy()

# Baseline: a long-form highlights video published 0-1 days BEFORE a match,
# in 2024Q1. Every coefficient is relative to that.
#
# Three references, each chosen to make the output readable:
#
#   highlights       large (n=310), so the baseline is precisely measured, and
#                    near the top on views - most content types read as "how far
#                    below highlights", though unclassified and feature come out
#                    level with it.
#   0-1 before       the floor of the cycle, so all nine cycle coefficients come
#                    out positive and read as lift over the worst window.
#   2024Q1 (implied) C() takes the alphabetically first quarter as reference, so
#                    every quarter coefficient reads as growth since the start of
#                    the window. Not chosen - just worth knowing what it is.
#
# The first two are named explicitly because C() otherwise picks the
# alphabetically first level, which for content_type is community (n=9).

# C(published_quarter) gives each quarter its own baseline, so the audience
# growth at 2025Q3 stops being attributed to whatever else sits in that period.

model = smf.ols(
    'log10_views ~ is_short + is_son'
    ' + C(content_type, Treatment(reference="highlights"))'
    ' + C(cycle_bin, Treatment(reference="0-1 before"))'
    ' + C(published_quarter)',
    data=model_df).fit()

# statsmodels silently drops rows with a NaN in any model column - here the
# offseason videos, where days_from_match is NaN.
print(f'rows {len(df)} -> {len(model_df)} after the age floor and tiny types '
      f'-> {int(model.nobs)} used in the model')

rows 1952 -> 1827 after the age floor and tiny types -> 1703 used in the model


In [52]:
# The same model with and without the quarter terms, on the same rows, to show what the quarter control actually changes.
BASE = ('log10_views ~ is_short + is_son'
        ' + C(content_type, Treatment(reference="highlights"))'
        ' + C(cycle_bin, Treatment(reference="0-1 before"))')

without_quarter = smf.ols(BASE, data=model_df).fit()
model           = smf.ols(BASE + ' + C(published_quarter)', data=model_df).fit()

pd.DataFrame({
    'without_quarter': [10 ** without_quarter.params['is_short[T.True]'],
                        10 ** without_quarter.params['is_son[T.True]'],
                        without_quarter.rsquared],
    'with_quarter':    [10 ** model.params['is_short[T.True]'],
                        10 ** model.params['is_son[T.True]'],
                        model.rsquared],
}, index=['shorts x', 'Son x', 'R²']).round(2)

,without_quarter,with_quarter
shorts x,3.71,4.01
Son x,36.47,6.56
R²,0.46,0.64


In [55]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            log10_views   R-squared:                       0.636
Model:                            OLS   Adj. R-squared:                  0.630
Method:                 Least Squares   F-statistic:                     104.3
Date:                Tue, 25 Aug 2026   Prob (F-statistic):               0.00
Time:                        16:20:21   Log-Likelihood:                -1113.2
No. Observations:                1703   AIC:                             2284.
Df Residuals:                    1674   BIC:                             2442.
Df Model:                          28                                         
Covariance Type:            nonrobust                                         
                                                                            coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------

In [ ]:
#Since views is in log10 space, this function converts to mulitipliers

def as_multipliers(model):
    ci = model.conf_int()
    table = pd.DataFrame({
        'multiplier': 10 ** model.params,
        'ci_low':     10 ** ci[0],
        'ci_high':    10 ** ci[1],
        'p_value':    model.pvalues,
    })
    is_quarter = table.index.str.startswith('C(published_quarter)')
    is_intercept = table.index == 'Intercept'
    return table[~is_quarter & ~is_intercept].round(3)

as_multipliers(model)

,multiplier,ci_low,ci_high,p_value
is_short[T.True],4.009,3.277,4.904,0.000
is_son[T.True],6.558,4.154,10.352,0.000
"C(content_type, Treatment(reference=""highlights""))[T.feature]",1.013,0.604,1.699,0.961
"C(content_type, Treatment(reference=""highlights""))[T.match_preview]",0.747,0.547,1.021,0.068
"C(content_type, Treatment(reference=""highlights""))[T.no_playlist]",0.821,0.656,1.027,0.084
"C(content_type, Treatment(reference=""highlights""))[T.podcast]",0.296,0.241,0.363,0.000
"C(content_type, Treatment(reference=""highlights""))[T.press_interview]",0.378,0.313,0.457,0.000
"C(content_type, Treatment(reference=""highlights""))[T.show]",0.449,0.352,0.573,0.000
"C(content_type, Treatment(reference=""highlights""))[T.unclassified]",1.034,0.721,1.483,0.856
"C(cycle_bin, Treatment(reference=""0-1 before""))[T.4+ before]",1.761,1.355,2.290,0.000


**Shorts and Son are the two big levers.** Shorts get ×4.0 the views of an otherwise identical long-form video, and a Son Spotlight video gets ×6.6. Both at p<0.001. Both are measured against videos published in the same quarter, so neither is picking up the audience growth from the signing.

**Within the hortizonal / live format, content type is a big lever** Against a highlights package: podcast ×0.30, press_interview ×0.38, show ×0.45. A podcast gets under a third of the views a highlights video gets, all at p<0.001.

**The match cycle matters less than it looked.** Every window beats the 24 hours before kickoff, but by similar amounts: ×1.88 for 0–1 days after, ×1.80 for 3–4 days after, ×1.76 for 4+ days before. There's no sharp post-match spike — what the data shows is that the day before a match is the weakest slot and everything else is roughly equivalent.

**R² of 0.636, but not predictive.** A large part of it is the ten quarter terms, which describe when a video was published rather than anything about the video itself. The model puts you in the right ballpark for a type of video and still won't predict any particular one.

### OLS: Engagement Rate on format, content type, and match-cycle position
Baseline: a long-form highlights video published 0–1 days before a match.

In [60]:
engagement_model = smf.ols(
    'engagement_rate ~ is_short + is_son'
    ' + C(content_type, Treatment(reference="highlights"))'
    ' + C(cycle_bin, Treatment(reference="0-1 before"))'
    ' + C(published_quarter)',
    data=model_df).fit()

print(engagement_model.summary())

                            OLS Regression Results                            
Dep. Variable:        engagement_rate   R-squared:                       0.213
Model:                            OLS   Adj. R-squared:                  0.200
Method:                 Least Squares   F-statistic:                     16.17
Date:                Tue, 25 Aug 2026   Prob (F-statistic):           3.40e-68
Time:                        16:25:29   Log-Likelihood:                 4159.8
No. Observations:                1703   AIC:                            -8262.
Df Residuals:                    1674   BIC:                            -8104.
Df Model:                          28                                         
Covariance Type:            nonrobust                                         
                                                                            coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------

**Shorts trade engagement for reach.** Shorts get 0.87 percentage points less engagement while getting ×4.0 the views. Podcasts do the opposite: +2.18 points on a third of the views. Judging a podcast on views, or a
highlights clip on comment rate, misreads both.

**Son videos are the exception — they get both.** +1.97 percentage points1, alongside ×6.6 the views. This is the only lever in the analysis where reach and engagement move in the same direction.

**The 24 hours after kickoff is the engagement low point.** −0.91 points against the day before a match. Most other cycle windows have intervals crossing zero, so the trough is the part of the timing story that
holds up on engagement.

**R² of 0.213 against 0.636 for views.** Engagement rate is less predictable from format, content type, and timing than reach is.

## Duration, within long-form

If the shorts format gets more views than the horizontal / live format, does that mean shorter duration videos always get more views? Even within horizontal / live content types like podcasts?

In [41]:
long_df = df[df['format'] != 'short'].copy()
long_df['dur_min']   = pd.to_timedelta(long_df['duration']).dt.total_seconds() / 60
long_df['log10_dur'] = np.log10(long_df['dur_min'].clip(lower=0.1))

BANDS  = [0, 1, 3, 10, 30, 60, np.inf]
LABELS = ['<1m', '1-3m', '3-10m', '10-30m', '30-60m', '60m+']
long_df['dur_band'] = pd.cut(long_df['dur_min'], bins=BANDS, labels=LABELS, right=False)

long_df.groupby('dur_band', observed=True).agg(
    n=('view_count', 'size'),
    median_views=('view_count', 'median'),
    median_engagement=('engagement_rate', 'median'))

,n,median_views,median_engagement
dur_band,,,
<1m,219,1652.0,0.049130
1-3m,208,1142.0,0.047425
3-10m,356,864.0,0.041185
10-30m,360,843.5,0.054025
30-60m,269,702.0,0.059370
60m+,12,423.5,0.031040


In [42]:
long_df.groupby('content_type')['dur_min'].agg(['size', 'median']).sort_values('median')

,size,median
content_type,,
match_preview,72,1.033333
highlights,264,1.050000
unclassified,96,1.175000
no_playlist,96,1.608333
community,9,3.083333
feature,20,3.808333
press_interview,338,8.366667
show,148,21.000000
podcast,372,38.166667


In [43]:
long_df.groupby(['content_type', 'dur_band'], observed=True).agg(
    n=('view_count', 'size'),
    median_views=('view_count', 'median'),
    median_engagement_rate=('engagement_rate', 'median'))

n  median_views  median_engagement_rate
content_type    dur_band                                           
community       1-3m        4         391.5                0.062245
                3-10m       5         933.0                0.066450
feature         <1m         6        3171.5                0.014750
                3-10m      14         636.5                0.037760
full_match      60m+        9         352.0                0.025310
highlights      <1m       112        1174.5                0.041420
                1-3m       70        1285.0                0.048215
                3-10m      79        5752.0                0.024870
                10-30m      3        4031.0                0.029270
match_preview   <1m        32        1680.0                0.056340
                1-3m       38         870.5                0.041290
                3-10m       1        3485.0                0.033290
                10-30m      1        1693.0                0.028940
no_playlist     <1m        23        2087.0                0.060660
                1-3m       33        2048.0                0.052120
                3-10m      23        3204.0                0.058360
                10-30m      8        1541.5                0.052425
                30-60m      6        3189.0                0.037970
                60m+        3         976.0                0.046110
podcast         <1m         1        1518.0                0.053360
                3-10m       1        2339.0                0.038480
                10-30m    123         769.0                0.066670
                30-60m    247         629.0                0.060330
press_interview <1m         3         336.0                0.059520
                1-3m       28         289.5                0.050365
                3-10m     154         379.5                0.045250
                10-30m    140        1127.0                0.048055
                30-60m     13        3949.0                0.051480
show            <1m         3        1202.0                0.038260
                1-3m        3        1236.0                0.039810
                3-10m      65        1466.0                0.044230
                10-30m     77         415.0                0.051040
unclassified    <1m        39       14746.0                0.059930
                1-3m       32        3599.0                0.053060
                3-10m      14        1236.0                0.040580
                10-30m      8        1907.0                0.051370
                30-60m      3        4123.0                0.048090

So even though it looked like there was an inverse relationship between duration and views, it turned out that was just misreading content types as duration. Some contents types are longer due to their conventions, but its their content type that is responsible for their views. Once you look inside the content type categories you can see that for some categories like podcasts, the longer episodes actually have longer views.

## Synthesis

Putting the two models together. The regressions held up, so now I chart the median values for views and engagement against each other.

In [44]:
# --- Synthesis: reach and engagement pull in opposite directions ---
summary = (df.groupby('content_type')
             .agg(n=('view_count', 'size'),
                  median_views=('view_count', 'median'),
                  median_eng=('engagement_rate', 'median'))
             .query('n >= 20')
             .reset_index())

fig = px.scatter(
    summary, x='median_views', y='median_eng',
    log_x=True,
    size='n', size_max=45,
    color='content_type',
    color_discrete_map=CONTENT_COLORS,
    text='content_type',
    title='Reach and engagement pull against each other',
    labels={'median_views': 'Median views (log)',
            'median_eng': 'Median engagement rate'},
    height=560,
)

fig.update_traces(textposition='top center')
fig.update_layout(showlegend=False)   # points are directly labelled

fig.update_xaxes(tickvals=[1_000, 10_000, 100_000],
                 ticktext=['1K', '10K', '100K'])
fig.update_yaxes(tickformat='.1%')

display(summary)
fig.show()


,content_type,n,median_views,median_eng
0,feature,20,1209.0,0.029695
1,highlights,310,2686.0,0.034785
2,match_preview,73,1294.0,0.045400
3,no_playlist,548,4713.5,0.049585
4,podcast,372,676.5,0.061065
5,press_interview,338,560.5,0.047340
6,show,148,866.0,0.047785
7,unclassified,125,16216.0,0.053360


## Conclusions

**Reach and engagement are different goals, and most levers trade one for the
other.** Shorts get ×4.0 the views and 0.87 points less engagement. Every content
type loses to highlights on views and beats it on engagement. The 24 hours after
kickoff is the views peak and the engagement trough. Judging a podcast on views,
or a highlights clip on comment rate, misreads both.

**The post-match bump is smaller than it looked.** ×1.88 on day one, ×1.59 on day
two, and still ×1.80 on days three to four. Every window beats the final 24 hours
before a match, which is the weakest slot in the cycle for views, and the
strongest for engagement.

**"Short-form wins" is about the Shorts surface, not about length.** Within
long-form the relationship reverses: a highlights package 5× longer gets
roughly 5× the views. The lever is *publish to the Shorts feed*, not *make
everything shorter*.

**One player is the largest effect in the data** - ×6.6 views after controls,
and the only lever that gains engagement too, +1.97 points at p<0.001. Whether that is player content
generally or Son specifically cannot be answered here: the one comparable
playlist has 9 videos in this window.

**On-field context does not move content performance.** Result, league position
and opponent strength are all null. The apparent home/away effect dissolved once
Son's away-heavy content was separated out.

### What would change these conclusions

- **The Shorts probe.** 86% of Shorts have no playlist, so `content_type` is
  structurally a long-form label system. The segment that drives reach is the
  one with no labels.
- **A second star player.** Every subject finding rests on 69 videos and one
  arrival window.
- **More seasons.** This is 2024-01 to 2026-08. The 2024 era break shows how
  fast this channel's behaviour changes.
- **Multiple comparisons.** Seven or eight models were run against one dataset.
  The two main models are the trustworthy part; single results just under
  p=0.05 elsewhere should be treated as exploratory until re-tested.